In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

# OpenAI API key
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "") 

# MongoDB connection string
MONGO_URI = os.getenv("MONGO_URI", "")

# MongoDB database to store the knowledge graph
MONGO_DB = os.getenv("MONGO_DB", "")  

# MongoDB collection to store the knowledge graph
MONGO_GRAPH_COLLECTION = os.getenv("MONGO_GRAPH_COLLECTION", "")

# MongoDB collection to store the vector embeddings
MONGO_VECTOR_COLLECTION = os.getenv("MONGO_VECTOR_COLLECTION", "") 

# MongoDB collection to store the vector index
MONGO_VECTOR_INDEX = os.getenv("MONGO_VECTOR_INDEX", "")  

In [3]:
from langchain_mongodb.vectorstores import MongoDBAtlasVectorSearch
from langchain_mongodb.graphrag.graph import MongoDBGraphStore
from langchain_openai import OpenAIEmbeddings
from langchain.chat_models import init_chat_model
from pymongo import MongoClient

chat_model = init_chat_model("gpt-4o", model_provider="openai", temperature=0, api_key=OPENAI_API_KEY)
embeddings = OpenAIEmbeddings(model="text-embedding-3-large", api_key=OPENAI_API_KEY)

client = MongoClient(MONGO_URI)

graph_store = MongoDBGraphStore(
    connection_string=MONGO_URI,
    database_name=MONGO_DB,
    collection_name=MONGO_GRAPH_COLLECTION,
    entity_extraction_model = chat_model,
)

vector_search = MongoDBAtlasVectorSearch(
    collection=client[MONGO_DB][MONGO_VECTOR_COLLECTION],
    embedding=embeddings,
    index_name=MONGO_VECTOR_INDEX,   # pre-created Atlas Vector Search index
    # text_key="text",                 # the field you used when adding docs
    embedding_key="embedding",       # the vector field in Atlas
)

In [4]:
from typing import List, Dict, Any, Tuple, Set

CHUNK_ID_META_KEY = "_id"

# -----------------------------------------------------------
# Vector search for top chunks
# -----------------------------------------------------------
def vector_search_chunks(
    vector_store: MongoDBAtlasVectorSearch,
    query: str,
    k: int = 6,
) -> List[Dict[str, Any]]:
    """
    Returns a list of LangChain Documents.
    Each doc typically has: .page_content (text) and .metadata (dict).
    Expect metadata[CHUNK_ID_META_KEY] to identify the corresponding graph node.
    """
    docs = vector_store.similarity_search(query, k=k)
    return [ {"text": d.page_content, "metadata": dict(d.metadata)} for d in docs ]


# -----------------------------------------------------------
# Graph expansion helpers
# -----------------------------------------------------------
def fetch_chunk_node_ids_from_docs(
    docs: List[Dict[str, Any]],
    chunk_id_key: str = CHUNK_ID_META_KEY
) -> List[str]:
    ids = []
    for d in docs:
        m = d.get("metadata", {})
        if chunk_id_key in m and m[chunk_id_key]:
            ids.append(str(m[chunk_id_key]))
    return ids


def subgraph_from_chunk_ids_raw_mongo(
    db,
    chunk_ids: List[str],
    max_depth: int = 1,
) -> Dict[str, Any]:
    """
    Fallback / robust MongoDB query that doesn’t rely on GraphStore convenience methods.
    Performs a breadth-first expansion up to `max_depth` hops from the chunk nodes.
    Returns {"nodes": [...], "edges": [...]}.
    """
    col = db[MONGO_GRAPH_COLLECTION]

    # Seed nodes = chunk nodes
    nodes_map: Dict[str, Dict] = {}
    edges_map: Dict[str, Dict] = {}

    # Fetch seed chunk nodes
    for cid in chunk_ids:
        node = col.find_one({"_id": cid})
        if node:
            nodes_map[str(node["_id"])] = node

    frontier: Set[str] = set(nodes_map.keys())

    for _ in range(max_depth):
        if not frontier:
            break

        next_frontier: Set[str] = set()

        # Find edges touching current frontier
        touching_edges = col.find({
            "type": {"$exists": True}, 
            "source": {"$in": list(frontier)}
        })
        for e in touching_edges:
            edges_map[str(e["_id"])] = e
            tgt_id = str(e["target"])
            # add target node
            node = col.find_one({"_id": tgt_id})
            if node and tgt_id not in nodes_map:
                nodes_map[tgt_id] = node
                next_frontier.add(tgt_id)

        touching_edges_rev = col.find({
            "type": {"$exists": True},
            "target": {"$in": list(frontier)}
        })
        for e in touching_edges_rev:
            edges_map[str(e["_id"])] = e
            src_id = str(e["source"])
            node = col.find_one({"_id": src_id})
            if node and src_id not in nodes_map:
                nodes_map[src_id] = node
                next_frontier.add(src_id)

        frontier = next_frontier

    return {
        "nodes": list(nodes_map.values()),
        "edges": list(edges_map.values())
    }


def subgraph_from_chunk_ids_graphstore(
    chunk_ids: List[str],
    max_depth: int = 1,
    allowed_relationship_types: List[str] = None,
):
    """
    If your GraphStore version exposes traversal helpers, use them here.
    We’ll attempt to call a generic 'subgraph_from_ids' if present;
    otherwise we fall back to raw Mongo queries above.
    """
    # Attempt to use a convenience method if available in your version.
    if hasattr(graph_store, "subgraph_from_ids"):
        return graph_store.subgraph_from_ids(
            ids=chunk_ids,
            max_depth=max_depth,
            allowed_relationship_types=allowed_relationship_types
        )
    return list(client[MONGO_DB][MONGO_GRAPH_COLLECTION].find({"attributes.vector_id": {"$in": chunk_ids}}
))

# -----------------------------------------------------------
# Main query function (Vector → Graph expansion)
# -----------------------------------------------------------
def graph_rag_query(
    query: str,
    k: int = 6,
    graph_max_depth: int = 1,
    allowed_relationship_types: List[str] = None,
    vec: MongoDBAtlasVectorSearch = vector_search,
    gstore: MongoDBGraphStore = graph_store,
) -> Dict[str, Any]:
    """
    1) Vector search to get top-k chunks (retrieval).
    2) Expand those chunk nodes into a subgraph (entities/edges) via GraphStore.
    3) Return a compact bundle for downstream answer generation.
    """

    # 1) Retrieve chunks by vector search
    top_docs = vector_search_chunks(vec, query, k=k)

    chunk_ids = fetch_chunk_node_ids_from_docs(top_docs, chunk_id_key=CHUNK_ID_META_KEY)

    # 2) Expand to subgraph (neighbors/entities/relations around those chunk nodes)
    subgraph = subgraph_from_chunk_ids_graphstore(
        chunk_ids=chunk_ids
    )

    # 3) Return context bundle (ready for LLM prompting or your own reranking)
    return {
        "query": query,
        "top_chunks": top_docs,
        "subgraph": subgraph,
        "params": {
            "k": k,
            "graph_max_depth": graph_max_depth,
            "allowed_relationship_types": allowed_relationship_types
        }
    }

user_query = "How much V-JEPA2 use gpus during training?"
result = graph_rag_query(
        query=user_query,
        k=5,
        graph_max_depth=2,
        vec=vector_search,
        gstore=graph_store
    )


In [5]:
result

{'query': 'How much V-JEPA2 use gpus during training?',
 'top_chunks': [{'text': 'Appendix\nA V-JEPA 2 Pretraining\nA.1 Pretraining Hyperparameters\nAs detailed in Section 2.4, our training pipeline consisted of two phases: 1) a constant learning rate phase and\n2) a cooldown phase. For all models, we trained in the first phase until we observed plateauing or diminishing\nperformance on the IN1K, COIN, and SSv2 tasks. At this point, we initiated the cooldown phase.\nTable 9 Pretraining Hyperparameters.Common parameters for pretraining large computer vision models. We\nreport these parameters for both the primary training phase and the cooldown phase.\nParameter Primary Phase Cooldown Phase\nNumber of frames 16 64\nFrames per Second 4.0 4.0\nCrop Size 256 [256, 384, 512]\nRandom Resize Aspect Ratio [0.75 1.35] [0.75, 1.35]\nRandom Resize Scale [0.3, 1.0] [0.3, 1.0]\nSteps Variable 12000\nWarmup Steps 12000 N/A\nBatch Size (global) 3072 3072\nStarting Learning Rate 1e-4 5.25e-4\nFinal Le

In [ ]:
import pandas as pd
import networkx as nx
from pyvis.network import Network

filter_out_attributes = ["authored_by", "authors", "vector_id"]
filter_out_types = ["Person"]

def visualize_result_graph(data, output_html="output.html"):
    palette = [
        "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
        "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
        "#393b79", "#637939", "#8c6d31", "#843c39", "#7b4173"
    ]
    
    df = pd.DataFrame(data)
    all_type = df["type"].unique()
    color_map = {c: palette[i % len(palette)] for i, c in enumerate(all_type)}

    def format_attributes(attrs):
        return "<br>".join(f"{k}: {', '.join(v)}" for k, v in attrs.items()) if attrs else ""
    
    G = nx.DiGraph()

    # Create nodes
    for doc in data:
        node_id = str(doc["_id"])
        info = f"Type: {doc.get('type', '')}"
        if doc.get('type', '') not in filter_out_types:
            if "attributes" in doc:
                attr_info = format_attributes(doc["attributes"])
                if attr_info:
                    info += "<br>" + attr_info
            
            G.add_node(node_id, label=node_id, title=info.replace("<br>", "\n"), color=color_map.get(doc["type"]),)

    # Create edges
    for doc in data:
        source = str(doc["_id"])
        rels = doc.get("relationships", {})
        targets = rels.get("target_ids", [])
        types = rels.get("types", [])
        attrs = rels.get("attributes", [])
        
        for i, target in enumerate(targets):
            edge_type = types[i] if i < len(types) else ""
            extra = attrs[i] if i < len(attrs) else {}
            edge_info = f"Relationship: {edge_type}"
            if extra:
                edge_info += "<br>" + format_attributes(extra)
            if edge_type not in filter_out_attributes:
                G.add_edge(source, str(target), label=edge_type, title=edge_info.replace("<br>", "\n"))

        node_attrs = doc.get("attributes", {})
        for sub_att, v_sub in node_attrs.items():
            # print(f"Source: {source}, Sub-attribute: {sub_att}, Values: {v_sub}")
            if sub_att not in filter_out_attributes:
                for v in v_sub:
                    G.add_edge(source, str(v), label=sub_att)

    # Build and configure network
    nt = Network(height="750px", width="100%", directed=True, notebook=True)
    nt.from_nx(G)
    nt.show(output_html)

In [7]:
visualize_result_graph(result["subgraph"], output_html="result_graph.html")

result_graph.html


In [8]:
from langchain_openai import ChatOpenAI

def generate_answer_with_llm(result: dict) -> str:
    query = result["query"]
    subgraph = result["subgraph"]
    top_chunks = result["top_chunks"]

    system_prompt = (
        "You are a helpful AI assistant that answers questions using graph-based retrieved knowledge. "
        "The following data consists of text chunks and a knowledge graph of entities and relationships. "
        "Use both the textual and relational information to generate an accurate, concise, and insightful answer."
    )

    user_prompt = f"""
    Question:
    {query}

    Relevant text chunks:
    {top_chunks}

    Knowledge graph context:
    {subgraph}

    Write your answer in a factual and coherent way, citing which concepts or entities from the graph support your reasoning when relevant.
    """

    llm = ChatOpenAI(model="gpt-4o", temperature=0.2)

    # Invoke model
    response = llm.invoke([
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ])

    return response.content

generate_answer_with_llm(result)


'The V-JEPA 2 model uses 512 H100 GPUs during its training process, specifically for Stage 2 and Stage 3 of its training setup. This information is supported by the text chunk that details the training setup, which mentions the use of 512 H100 GPUs along with a global batch size of 2048 for Stage 2 and 1024 for Stage 3. This setup is part of the model\'s training using Pytorch 2.5.1 and the Perception LM training code, which has been modified with the V-JEPA 2 encoder. The use of H100 GPUs is also confirmed in the knowledge graph under the "H100 GPUs" entity, which specifies their use in these training stages.'

In [9]:
!open "result_graph.html"